In [17]:
from langchain_groq import ChatGroq # for LLM
from langchain_huggingface.embeddings import HuggingFaceEmbeddings # for embeddings model
from langchain.tools import tool # for custome tools
from langchain_community.document_loaders.csv_loader import CSVLoader # for loading csv file
from langchain_text_splitters import RecursiveCharacterTextSplitter # for chunking data
from langchain_qdrant import QdrantVectorStore # for vector db
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, AIMessage # for messaging type
from langgraph.graph.message import add_messages # for message adding
from langchain_tavily import TavilySearch # for web search
from pydantic import BaseModel, Field # for structured output
from langgraph.checkpoint.memory import InMemorySaver # for configuration
# for langgraph workflow
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, List, Literal
import os
# env
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [ ]:
llm = ChatGroq(model="moonshotai/kimi-k2-instruct-0905", api_key=os.getenv("GROQ_API_KEY"))
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [3]:
class GreetingClassification(BaseModel):
    category: Literal["greeting", "non_greeting"] = Field(
        description="Whether the message is a greeting or non-greeting"
    )

In [4]:
class AnswerSatisfactionClassification(BaseModel):
    category: Literal["yes", "no"] = Field(
        description="Whether the message is yes or no"
    )

In [5]:
structured_llm = llm.with_structured_output(GreetingClassification, method="json_mode")
structured_llm_answer = llm.with_structured_output(AnswerSatisfactionClassification, method="json_mode")

In [6]:
all_docs = []

for csv_path in ["../datasets/data_preprocessed/device_manuals.csv", "../datasets/data_preprocessed//question_answer.csv"]:
    if os.path.exists(csv_path):
        loader = CSVLoader(file_path=csv_path, encoding="latin1")
        docs = loader.load()  # Each row = 1 Document
        all_docs.extend(docs)  # Keep Document objects

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
texts = text_splitter.split_documents(all_docs)

In [ ]:
url = os.getenv("QDRANT_DB_URL")
qdrant = QdrantVectorStore.from_documents(
    texts,
    embeddings,
    url=url,
    api_key=os.getenv("QDRANT_API_KEY"),
    prefer_grpc=True,
    collection_name="project",
)

In [9]:
query = "Who is at risk for Lymphocytic Choriomeningitis (LCM)?"
found_docs = qdrant.similarity_search(query, k=5)
found_docs

[Document(metadata={'row': 2, 'source': '../datasets/data_preprocessed//question_answer.csv', '_id': '33b7d169-e04e-4d76-8732-13c09c2465f8', '_collection_name': 'project'}, page_content='question_answer: Question: Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?. Answer: Individuals of all ages who come into contact with urine, feces, saliva, or blood of wild mice are potentially at risk for infection. Owners of pet mice or hamsters may be at risk for infection if these animals originate from colonies that were contaminated with LCMV, or if their animals are infected from other wild mice. Human fetuses are at risk of acquiring infection vertically from an infected mother. \n                \nLaboratory workers who work with the virus or handle infected animals are also at risk. However, this risk can be minimized by utilizing animals from sources that regularly test for the virus, wearing proper protective laboratory gear, and following appropriate safety precautions..'),
 Docu

In [18]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_DB_URL"), 
    api_key=os.getenv("QDRANT_API_KEY")
)

print(qdrant_client.get_collections())

qdrant_vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name="project",  # Your collection name
    embedding=embeddings
)

collections=[CollectionDescription(name='project')]


In [21]:
query = "What are rabies?"
found_docs = qdrant_vectorstore.similarity_search(query, k=5)
found_docs

[Document(metadata={'source': '../datasets/data_preprocessed//question_answer.csv', 'row': 4861, '_id': '8be9f03d-77cc-4b28-aff1-0afb182200ec', '_collection_name': 'project'}, page_content="question_answer: Question: What is (are) Rabies ?. Answer: Rabies is a deadly animal disease caused by a virus. It can happen in wild animals, including raccoons, skunks, bats and foxes, or in dogs, cats or farm animals. People get it from the bite of an infected animal.     In people, symptoms of rabies include fever, headache and fatigue, then confusion, hallucinations and paralysis. Once the symptoms begin, the disease is usually fatal. A series of shots can prevent rabies in people exposed to the virus. You need to get them right away. If an animal bites you, wash the wound well; then get medical care.     To help prevent rabies       -  Vaccinate your pet. Rabies vaccines are available for dogs, cats and farm animals    -  Don't let pets roam    -  Don't approach stray animals. Animals with rab

In [22]:
qdrant_client.delete_collection(collection_name="project")

True